In [1]:
#URL : https://www.kaggle.com/competitions/sentence-classification-skill-pixel-voai-2026/overview

In [1]:
#ext libraries
import os
import pandas as pd
import numpy as np
import re
import unicodedata
import warnings

#NLP libraries
import nltk
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords
from nltk import FreqDist


from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import FunctionTransformer
from sklearn.pipeline import Pipeline
from sklearn.model_selection import cross_validate,cross_val_score,KFold
from sklearn.metrics import f1_score,accuracy_score
from sklearn.feature_extraction.text import TfidfVectorizer,CountVectorizer

from sklearn.linear_model import LogisticRegression
from xgboost import XGBClassifier

#DL libraries
import tensorflow as tf
from tensorflow import keras
from keras import layers
from tensorflow.keras.callbacks import EarlyStopping


nltk.download('punkt')
nltk.download('stopwords')
nltk.download('punkt_tab')

warnings.simplefilter(action = 'ignore')

[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\Administrator\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\Administrator\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package punkt_tab to
[nltk_data]     C:\Users\Administrator\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


In [31]:
VOCAB_SIZE = 10000

In [41]:
sample_data = {
    'Text': [
        "This movie was absolutely fantastic! I loved every moment.",
        "Terrible experience. Would not recommend to anyone.",
        "The product is okay, nothing special but works fine.",
        "AMAZING service! The staff was so helpful and friendly!",
        "Not exactly thrilled with the purchase. It broke after a week.",
        "Great quality and fast shipping. Very satisfied!",
        "Worst purchase ever. Complete waste of money.",
        "The food was good but the service was terrible.",
        "I do not like this at all. Very disappointed.",
        "Exceeded my expectations! Highly recommend!",
        "Meh, it's alright I guess.",
        "Perfect! Exactly what I was looking for.",
        "Not bad, but could be better.",
        "Horrible quality. Don't buy this!",
        "Love it! Will definitely buy again.",
        "The worst experience I've ever had.",
        "Great, another delay. Just what I needed.",  # Sarcasm example
        "Good food, terrible service.",  # Mixed sentiment
        "Fantastic product with amazing features!",
        "Complete disappointment. Avoid at all costs."
    ],
    'Label': [1, 0, 1, 1, 0, 1, 0, 0, 0, 1, 1, 1, 1, 0, 1, 0, 0, 0, 1, 0]  # 1=Positive, 0=Negative
}

In [42]:
X_train = pd.DataFrame(sample_data)
#X_train = pd.read_csv('https://huggingface.co/datasets/NHANGIOI/Learning-Training/resolve/main/sentence-classification-skill-pixel-voai-2026/train.csv')
y_train = X_train['Label']
X_train.drop(columns = 'Label',inplace = True)
X_train = pd.Series(X_train.squeeze())

In [43]:
def normalize_text(seq,lower_case = True,rev_punctuation = True,replace_nums = True):
    seq = unicodedata.normalize('NFKC',seq)
    seq = ' '.join(seq.split())
    
    if replace_nums == True:
        seq = re.sub(r"\d+",'<NUM>',seq)
    if rev_punctuation == True:
        seq = re.sub(r"[^\w\s!?,.]",'',seq)
    if lower_case == True:
        seq = seq.lower()
    
    return seq
def preprocess(seq,window_state = 3):
    seq = normalize_text(seq)
    token_list = nltk.word_tokenize(seq)
    
    negative_words = {'not','no','never','neither',"n't",'nobody','nothing'}
    contrast_words = {'but','and','however','moreover','therefore','although','because','so','since'}
    punctuation_words = {'!','?',',','.'}
    
    res = []
    negative_count = 0
    for token in token_list:
        if token.lower() in negative_words or token.lower().endswith("n't"):
            res.append(token)
            negative_count = window_state
        elif token in contrast_words or token in punctuation_words:
            res.append(token)
            negative_count = 0
        elif negative_count > 0:
            res.append(f"NOT_{token}")
            negative_count -= 1
        else:
            res.append(token)
    return ' '.join(res)

In [44]:
full_preprocess = Pipeline(steps = [
    ('preprocess',FunctionTransformer(lambda X:X.apply(preprocess),validate = False)),
    ('tfidf',TfidfVectorizer(
        ngram_range = (1,2),
        min_df = 2,
        max_df = 0.85,
        sublinear_tf = True
    ))
])

In [45]:
X_all = full_preprocess.fit_transform(X_train).toarray().astype("float32")
y_all = y_train.to_numpy().astype("float32")
input_dim = X_all.shape[1]
print("X_tr shape:", X_all.shape)

X_tr shape: (20, 25)


In [ ]:
model = keras.Sequential([
    layers.Input(shape = (input_dim,)),
    
    layers.Dense(units = 1024),
    layers.BatchNormalization(),
    layers.Activation('relu'),
    layers.Dropout(0.25),
    
    layers.Dense(units = 512),
    layers.BatchNormalization(),
    layers.Activation('relu'),
    layers.Dropout(0.25),

    layers.Dense(units = 256),
    layers.BatchNormalization(),
    layers.Activation('relu'),
    layers.Dropout(0.25),
    
    layers.Dense(units = 1,activation = 'sigmoid')
])

In [47]:
model.compile(
    optimizer = 'adam',
    loss = 'binary_cross_entropy',
    metrics = ['accuracy']
)

In [48]:
history = model.fit(
    X_all,
    y_all,
    epochs = 10,
)

Epoch 1/10


ValueError: Could not interpret loss identifier: binary_cross_entropy

In [ ]:
X_test_1 = pd.Series(["You are good",
                      "Fuck you",
                      "nigger",
                      "I hate you",
                      "I love you 3000!!!",
                      "I want swallow your ball"])
preds = model.predict(X_test_1)
print(preds)

[1 0 1 0 1 1]


In [23]:
X_test = pd.read_csv('https://huggingface.co/datasets/NHANGIOI/Learning-Training/resolve/main/sentence-classification-skill-pixel-voai-2026/test.csv',index_col = 'ID')
X_test = X_test.squeeze()
print(X_test.shape)

(38000,)


In [24]:
preds = model.predict(pd.Series(X_test))
submission = pd.DataFrame({
    'ID' : X_test.index,
    'Label' : preds
})
submission.to_csv('submission.csv',index = False)